# Model Comparison - All Methods

This notebook compares performance of KNN, Random Forest, and Multilayer Perceptron on the TEST SET.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load test data
test_df_encoded = pd.read_csv('../../data/processed/training/test_data_v0.csv')

X_test = test_df_encoded.drop('Price', axis=1).values
y_test = test_df_encoded['Price'].values

print(f"Test set shape: {X_test.shape}")
print(f"Test samples: {len(y_test)}")

In [ ]:
# ==================== LOAD ALL MODELS ====================
print("\n" + "="*80)
print("LOADING TRAINED MODELS")
print("="*80)

# Load KNN Model
try:
    knn_model = joblib.load('knn_model.pkl')
    knn_scaler = joblib.load('knn_scaler.pkl')
    X_test_scaled_knn = knn_scaler.transform(X_test)
    y_knn_pred = knn_model.predict(X_test_scaled_knn)
    print("✓ KNN model loaded")
except Exception as e:
    print(f"✗ KNN model load failed: {e}")
    y_knn_pred = None

# Load Random Forest Model
try:
    rf_model = joblib.load('rf_model.pkl')
    y_rf_pred = rf_model.predict(X_test)
    print("✓ Random Forest model loaded")
except Exception as e:
    print(f"✗ Random Forest model load failed: {e}")
    y_rf_pred = None

# Load MLP Model
try:
    mlp_model = joblib.load('mlp_model.pkl')
    mlp_scaler = joblib.load('mlp_scaler.pkl')
    X_test_scaled_mlp = mlp_scaler.transform(X_test)
    y_mlp_pred = mlp_model.predict(X_test_scaled_mlp)
    print("✓ MLP model loaded")
except Exception as e:
    print(f"✗ MLP model load failed: {e}")
    y_mlp_pred = None

# ==================== CALCULATE METRICS FOR ALL MODELS ====================
print("\n" + "="*80)
print("CALCULATING TEST SET METRICS FOR ALL MODELS")
print("="*80)

results = {}

# Define metric calculation function
def calculate_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R²': r2
    }

# Calculate for KNN
if y_knn_pred is not None:
    results['KNN'] = calculate_metrics(y_test, y_knn_pred, 'KNN (k=7)')
    print(f"\nK-Nearest Neighbors (k=7):")
    print(f"  MSE:  {results['KNN']['MSE']:,.2f}")
    print(f"  RMSE: ${results['KNN']['RMSE']:.2f}")
    print(f"  MAE:  ${results['KNN']['MAE']:.2f}")
    print(f"  MAPE: {results['KNN']['MAPE']:.2f}%")
    print(f"  R²:   {results['KNN']['R²']:.4f}")

# Calculate for Random Forest
if y_rf_pred is not None:
    results['RF'] = calculate_metrics(y_test, y_rf_pred, 'Random Forest')
    print(f"\nRandom Forest:")
    print(f"  MSE:  {results['RF']['MSE']:,.2f}")
    print(f"  RMSE: ${results['RF']['RMSE']:.2f}")
    print(f"  MAE:  ${results['RF']['MAE']:.2f}")
    print(f"  MAPE: {results['RF']['MAPE']:.2f}%")
    print(f"  R²:   {results['RF']['R²']:.4f}")

# Calculate for MLP
if y_mlp_pred is not None:
    results['MLP'] = calculate_metrics(y_test, y_mlp_pred, 'MLP Neural Network')
    print(f"\nMultilayer Perceptron:")
    print(f"  MSE:  {results['MLP']['MSE']:,.2f}")
    print(f"  RMSE: ${results['MLP']['RMSE']:.2f}")
    print(f"  MAE:  ${results['MLP']['MAE']:.2f}")
    print(f"  MAPE: {results['MLP']['MAPE']:.2f}%")
    print(f"  R²:   {results['MLP']['R²']:.4f}")

print("\n" + "="*80)

In [ ]:
# ==================== CREATE COMPARISON TABLE ====================
print("\n" + "="*80)
print("MODEL COMPARISON TABLE - TEST SET RESULTS")
print("="*80)

# Create DataFrame from results
if len(results) > 0:
    comparison_df = pd.DataFrame([results[key] for key in results.keys()])
    
    # Format the display
    display_df = comparison_df.copy()
    display_df['MSE'] = display_df['MSE'].apply(lambda x: f"{x:,.0f}")
    display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f"${x:.2f}")
    display_df['MAE'] = display_df['MAE'].apply(lambda x: f"${x:.2f}")
    display_df['MAPE'] = display_df['MAPE'].apply(lambda x: f"{x:.2f}%")
    display_df['R²'] = display_df['R²'].apply(lambda x: f"{x:.4f}")
    
    print("\n" + display_df.to_string(index=False))
    print("\n" + "="*80)
    
    # Determine best model for each metric
    print("\n🏆 BEST PERFORMERS BY METRIC:")
    print("="*80)
    
    results_numeric = pd.DataFrame([results[key] for key in results.keys()])
    
    best_mse_idx = results_numeric['MSE'].idxmin()
    best_rmse_idx = results_numeric['RMSE'].idxmin()
    best_mae_idx = results_numeric['MAE'].idxmin()
    best_mape_idx = results_numeric['MAPE'].idxmin()
    best_r2_idx = results_numeric['R²'].idxmax()
    
    print(f"✓ Best MSE:   {results_numeric.iloc[best_mse_idx]['Model']:20s} ({results_numeric.iloc[best_mse_idx]['MSE']:>12,.0f})")
    print(f"✓ Best RMSE:  {results_numeric.iloc[best_rmse_idx]['Model']:20s} (${results_numeric.iloc[best_rmse_idx]['RMSE']:>11.2f})")
    print(f"✓ Best MAE:   {results_numeric.iloc[best_mae_idx]['Model']:20s} (${results_numeric.iloc[best_mae_idx]['MAE']:>11.2f})")
    print(f"✓ Best MAPE:  {results_numeric.iloc[best_mape_idx]['Model']:20s} ({results_numeric.iloc[best_mape_idx]['MAPE']:>11.2f}%)")
    print(f"✓ Best R²:    {results_numeric.iloc[best_r2_idx]['Model']:20s} ({results_numeric.iloc[best_r2_idx]['R²']:>11.4f})")
    
    # Overall score (lower is better for MSE, RMSE, MAE, MAPE; higher is better for R²)
    print("\n" + "="*80)
    print("📊 OVERALL RANKING (by combined performance):")
    print("="*80)
    
    # Normalize metrics for ranking (0-100 scale)
    scores = pd.DataFrame(index=results_numeric.index)
    scores['Model'] = results_numeric['Model']
    
    # Lower is better metrics
    scores['MSE_score'] = 100 - ((results_numeric['MSE'] - results_numeric['MSE'].min()) / (results_numeric['MSE'].max() - results_numeric['MSE'].min()) * 100)
    scores['RMSE_score'] = 100 - ((results_numeric['RMSE'] - results_numeric['RMSE'].min()) / (results_numeric['RMSE'].max() - results_numeric['RMSE'].min()) * 100)
    scores['MAE_score'] = 100 - ((results_numeric['MAE'] - results_numeric['MAE'].min()) / (results_numeric['MAE'].max() - results_numeric['MAE'].min()) * 100)
    scores['MAPE_score'] = 100 - ((results_numeric['MAPE'] - results_numeric['MAPE'].min()) / (results_numeric['MAPE'].max() - results_numeric['MAPE'].min()) * 100)
    
    # Higher is better metrics
    scores['R2_score'] = ((results_numeric['R²'] - results_numeric['R²'].min()) / (results_numeric['R²'].max() - results_numeric['R²'].min()) * 100)
    
    # Calculate average score (emphasize MAE and MAPE as primary metrics)
    scores['Overall_Score'] = (scores['MAE_score'] * 0.30 + 
                                scores['MAPE_score'] * 0.30 + 
                                scores['RMSE_score'] * 0.20 + 
                                scores['R2_score'] * 0.15 + 
                                scores['MSE_score'] * 0.05)
    
    scores_sorted = scores.sort_values('Overall_Score', ascending=False)
    
    for idx, (i, row) in enumerate(scores_sorted.iterrows(), 1):
        medal = ['🥇', '🥈', '🥉'][idx-1] if idx <= 3 else '  '
        print(f"{medal} #{idx}: {row['Model']:20s} - Score: {row['Overall_Score']:.1f}/100")
    
    print("\n" + "="*80)
    print("\n✓ Comparison complete! See detailed results above.")